# LAB 2: Prognosis and survival functions

In this lab, we will analyze observations from different ovarian cancer patients and we will create different survival functions employing univariate (Kaplan-Meier estimate) and multivariate (CPH and Extra Survival Trees) analysis.

Remember to copy this notebook into your own drive and your JHED ID as suffix (It's the part before the @ symbol in your email, not your Hopkins ID in the SIS):


*   eg: Lab2_Prognosis_myjhedID

Please always remember to use the MLMA coding rubric.

In [ ]:
# Install the package you need for this lab
# You may need to install them every time you restart the runtime
!pip install lifelines
!pip install xlrd
!pip install scikit-survival

In [ ]:
# Feel free to add more libraries if you need them
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lifelines import CoxPHFitter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sksurv.ensemble import ExtraSurvivalTrees
from sksurv.preprocessing import OneHotEncoder
import xlrd
import zipfile

**DATA**

The data that we are going to use in this lab is a subset of the data described in the paper ["Gene Expression Profile for Predicting Survival in Advanced-Stage Serous Ovarian Cancer Across Two
Independent Datasets"](https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0009615). According to the authors:

>One hundred ten Japanese patients who were diagnosed with advanced-stage serous ovarian cancer between July 1997 and June 2008 were included in this study. Fresh-frozen samples were obtained from primary tumor tissues during primary debulking surgery prior to chemotherapy. [...] frozen tissues containing more than 80% of tumor cells upon histological evaluation were used for RNA extraction. [...] Optimal debulking surgery was defined as ≤ 1cm of gross residual disease. [...] Overall survival time was calculated as the interval from primary surgery to the death due to ovarian cancer.

For our purposes, we are going to consider time-to-death, debulking optimality, and five biomarker genes associated with ovarian cancer survival (CXCL12, NCOA3, PDPN, TEAD1, YWHAB).

In [ ]:
# Load the data (adjust path if necessary)
df=pd.read_csv('./data.csv')

NameError: name 'pd' is not defined

In [ ]:
# Visualize the data and obtain statistics (number of observations and predictor variables)
print('Number of observations: ', len(df))
print('Number of predictor variables: ', len(df.columns)-1)
df.head()

## Task 1. Check your data (3 POINTS)
---
Please check the debulking optimality and CXCL12 gene expression distributions of the dataset

In [ ]:
# Please plot a histogram of the debulking optimality distribution:
# debulk:0 - not optimal
# debulk:1 - optimal

# YOUR CODE GOES HERE

# and answer a question briefly：how many patients in this dataset expressed the NCOA3 gene (NCOA3>0)?


# Please plot a histogram of the CXCL12 gene expression distribution:
# The numeric values represent a patient's gene expression intensity, with high values indicating high levels of expression.

# YOUR CODE GOES HERE

: 

## Task 2 (2 POINT)
---
Do you think the dataset we are using is balanced in terms of CXCL12 gene expression? Why or why not?

Answer:

# 1. Prognostic Model

Different from survival models, prognostic models try to answer questions referred to fixed period or related to a certain event.

## TASK 3 (4 POINTS)
---
The maximum follow-up time for patients in was 2430 days. Answer the following quesitons briefly (without repeating any answers).

1. Patient GSM432223 is listed as having time = 60 days and death = 0. What is a potential conclusion you can draw about their outcome?

Answer:

2. Patient GSM432229 is listed as having time = 1710 days and death = 1. What is a potential conclusion you can draw about their outcome?

Answer:

3. Patient GSM432307 is listed as having time = 2430 days and death = 0. What is a potential conclusion you can draw about their outcome?

Answer:

4. Assuming that your conclusions are all true, which one of the above patients is considered censored?

Answer:

## TASK 4 (2 POINTS)
---
To build a prognostic model (for instance, a model that predicts if someone will die at or before 90 days after surgical intervention for ovarian cancer), we should always delete all or part of the censored data. Briefly expain why this is necessary.

Answer:

# 2. Univariate analysis: Univariate survival functions and cumulative hazard

## TASK 5 (5 POINTS)

What is the probablity of a patient being alive after 1500 days? i.e. Calculate $P_s(1500)$

In [ ]:
# YOUR CODE GOES HERE

## TASK 6 (10 POINTS)
---

Calculate and plot the survival function employing the Kaplan-Meier estimate  

In [ ]:
# Calculate and plot the survival function employing the Kaplan-Meier estimate
def calculate_survival_KM(df, TimeColumn, EventColumn, EVENT_HAPPEN_INDICATOR, EVENT_NOT_HAPPEN_INDICATOR):
  """
Generates a dataframe containing the survival function (time vs probability of survival)
employing the Kaplan-Meier estimate.
Inputs:
      df: input dataframe
      TimeColumn: name of the colum containing the event time
      EventColumn: name of the column indicating if the event happened HAPPEN_INDICATOR or not NOT_HAPPEN_INDICATOR
      EVENT_HAPPEN_INDICATOR: the indicator status when event happens, like 1 for death
      EVENT_NOT_HAPPEN_INDICATOR: the indicator status when event does not happen, like 0 for alive
Outputs:
      dfSurv: dataframe containing the time and probability of survival
  """

  # YOUR CODE GOES HERE

  return dfSurv


def plot_survival(df, TimeColumn, EventColumn, Name, EVENT_HAPPEN_INDICATOR, EVENT_NOT_HAPPEN_INDICATOR):
  """
  This function plots the survival function of the whole df. This function calls
  calculate_survival_KM.
  Inputs:
      df: input dataframe
      TimeColumn: name of the colum containing the event time
      EventColumn: name of the column indicating if the event happened (1) or not (0)
      Name: legend to display, associated to the survival function
      EVENT_HAPPEN_INDICATOR: the indicator status when event happens, like 1 for death
      EVENT_NOT_HAPPEN_INDICATOR: the indicator status when event does not happen, like 0 for alive

  """
  # YOUR CODE GOES HERE

In [ ]:
# Plot the survival function obtained with the entire dataframe
TimeColumn='time'
EventColumn='death'
plot_survival(df, TimeColumn, EventColumn, 'All', 1, 0)

## TASK 7 (10 POINTS)
---
Code a function that displays the survival function of two categories of a certain predictor (e.g. ANKRD27 expression > 0 vs. ANKRD27 expression ≤ 0) in the same plot.

In [ ]:
# function to print two categories
def plot_survival_twocat(df, predictor, cat1, cat0,cat1_name, cat0_name ):
  """
  Displays the survival function of two categories of a certain predictor
  (eg. optimal vs. nonoptimal debulking, NCOA3 expression > 1 vs. NCOA3 expression ≤ 1) in the same plot. This function
  only considers binary categories: 0 and 1.
  Inputs:
        df: input dataframe
        predictor: column containing the predictor variable that we want to study
        cat1: Label of the category associated to one category (eg. '0')
        cat0: Label of the category associated to the other category (eg. '1')
        cat1_name: Name for cat1's label (eg. 'debulked')
        cat0_name: Name for cat0's label (eg. 'not_debulked')
  """
  # YOUR CODE GOES HERE

In [ ]:
# Plot survival for tumor size ≥ 1 cm vs. tumor size < 1
predictor = 'debulk'
cat1 = 1
cat0 = 0
cat1_name = 'optimal'
cat0_name = 'nonoptimal'
plot_survival_twocat(df, predictor, cat1, cat0,cat1_name,cat0_name )

In [ ]:
# Plot survival for ANKRD27 expression > 0 vs. ANKRD27 expression ≤ 0

# YOUR CODE GOES HERE

In [ ]:
# Plot survival for CXCL12 expression > 0 vs. CXCL12 expression ≤ 0

# YOUR CODE GOES HERE

Which predictor above influences the survival probability in the first 500 days the least?

Answer:

## TASK 8 (10 POINTS)
---
Plot the cumulative Hazard using the Nelson-Aalen estimator

In [ ]:
# Plot the cumulative Hazard using the Nelson-Aalen estimator
def cumulative_hazard_NA(df, TimeColumn, EventColumn, EVENT_HAPPEN_INDICATOR, EVENT_NOT_HAPPEN_INDICATOR):
  """
  Generates a dataframe containing the cumulative hazard  (H(t)) employing the
  Nelson-Aalen estimator.
  Inputs:
        df: input dataframe
        TimeColumn: name of the colum containing the event time
        EventColumn: name of the column indicating if the event happened or not
        EVENT_HAPPEN_INDICATOR: the indicator status when event happens, like 1 for death
        EVENT_NOT_HAPPEN_INDICATOR: the indicator status when event does not happen, like 0 for being alive
  Outputs:
        dfH: dataframe containing the time and Cumulative Hazard (H(t))
  """

  # YOUR CODE GOES HERE

  return dfH

def plot_cumH(df, TimeColumn, EventColumn, Name,EVENT_HAPPEN_INDICATOR, EVENT_NOT_HAPPEN_INDICATOR):
  """
  This function plots the cumulative hazard of the whole df. This function calls
  calculate_hazard_NA.
  Inputs:
      df: input dataframe
      TimeColumn: name of the colum containing the event time
      EventColumn: name of the column indicating if the event happened (1) or not (0)
      Name: legend to display, associated to the CH
      EVENT_HAPPEN_INDICATOR: the indicator status when event happens, like 1 for death
      EVENT_NOT_HAPPEN_INDICATOR: the indicator status when event does not happen, like 0 for being alive
  """

  # YOUR CODE GOES HERE

In [ ]:
# Plot the cumulative hazard of df
TimeColumn='time'
EventColumn='death'
plot_cumH(df, TimeColumn, EventColumn, 'All', 1, 0)

# 2. Multivariate analysis
**Cox Proportional Hazards**

Now, we are going to employ multivariate analysis using CPH to determine the survival funtions and the hazard ratio of different patients.
We will employ [lifelines](https://lifelines.readthedocs.io/en/latest/index.html), a python survival analysis library. The relevant packages have been imported for you at the beginning.

## TASK 9 (5 POINTS)
---
Divide the dataframe into 80% training and 20% testing subsets and normalize the data.

In [ ]:
#Divide the data into dfTrain 80% and dfTest 20%:
#And normalize the data
# YOUR CODE GOES HERE

## TASK 10 (5 POINTS)
---
Calculate and print the Cox proportional hazards coefficients of the training subset (dfTrain). As the data was normalized, the coefficients will provide information about the magnitude of the effect of each predictor variables. You can use CoxPHFitter from lifelines to do that.

In [ ]:
# Calculate and print the Cox proportional hazards coefficients

# YOUR CODE GOES HERE

In [ ]:
cph.print_summary()
cph.plot

In [ ]:
cph.params_.values

## TASK 11 (5 POINTS)
---
Plot the survival function of the three first participants in the test subset.

In [ ]:
# Plot the survival function of the three first participants in the test subset
participant1=0
participant2=1
participant3=2

# YOUR CODE GOES HERE

What is the most important property of CPH that you observe in the plot in terms of the shape of the three lines? (You can say more but we are looking for one key word)

Answer:

## TASK 12 (10 POINTS)
---
Create a function that calculates the hazard ratio between two patients. This tells us which one of the two patients is at a higher risk.

In [ ]:
# Create a function that calculates the hazard ratio between two patients.
# This will tell us which patient is at a higher risk.
def hazard_ratio(df, participant1, participant2, cph):
    '''
    Calculates the hazard ratio of participant1 vs participant 2 using
    the coefficients of the CPH model.

    Inputs:
        df: dataframe containing all the participants of the study subset
        participant1: index of the first participant in the df.
        participant2: index of the second participant in the df.
        cph: CPH model obtained with lifelines
    Outputs:
        HR: hazard ratio of participant 1 vs participant 2
    '''

    # YOUR CODE GOES HERE

    return HR

In [ ]:
print('Participant 1 vs participant 2 Hazard Ratio: ' + str(hazard_ratio(dfTest, 5, 10, cph)))

## TASK 13 (15 POINTS)
---
Create a function to calculate the concordance index of the CPH model you just trained, using the test data.

In [ ]:
# concordance index function
def c_index_mlma(df, TimeColumn, EventColumn, cph):
    '''
    Computes the C-Index employing the testing data and trained models.

   Inputs:
      df: input dataframe
      TimeColumn: name of the colum containing the event time
      EventColumn: name of the column indicating if the event happened (1) or not (0)
      cph: CPH model obtained with lifelines
  Outputs:
      CI: Concordance Index
    '''


    """
     y_true (array): array of true event times
        scores (array): model risk scores
        event (array): indicator, 1 if event occurred at that index, 0 for censorship
    """

    # YOUR CODE GOES HERE

    return CI

In [ ]:
# Concordance Index of the training data
TimeColumn='time'
EventColumn='event'
CI=c_index_mlma(dfTrain, TimeColumn, EventColumn, cph)
print(CI)

In [ ]:
# Your output above be similar to the CI calculated by the lifelines library (+/- 0.015)
print(cph.concordance_index_)

In [ ]:
# Concordance Index of the testing data

# YOUR CODE GOES HERE

# Survival Trees

Now, we are going to use a different type of multivariate analysis to calculate survival functions: ExtraSurvivalTrees. To perform this type of analysis, we will employ the [scikit-survival library](https://scikit-survival.readthedocs.io/en/stable/index.html).


## TASK 14 (5 POINTS)
---
Train a estimator using the training subset .

In [ ]:
# convert_data_survival function
def convert_data_survival(df):
    '''
    Preprocesses a DataFrame for survival analysis by
    applying one-hot encoding to categorical features (excluding 'time' and 'death')
    and then formats the 'death' (as boolean) and 'time' columns into
    a structured NumPy record array

    Inputs:
        df: Input DataFrame with survival data.

    Outputs:
        X: A DataFrame with one-hot encoded features.
        y: A structured array containing death as boolean and time.
    '''

    # YOUR CODE GOES HERE

    return X, y

In [ ]:
X_train, Y_train=convert_data_survival(dfTrain)
X_test,  Y_test=convert_data_survival(dfTest)

In [ ]:
estimator = ExtraSurvivalTrees().fit(X_train, Y_train)
estimator

## TASK 15 (5 POINTS)
---
Calculate the CI of the model when applied to the testing subset. Use the scikit-survival scoring functions to obtain the index. Compare this result with the CI obtained with the CPH models.

In [ ]:
# YOUR CODE GOES HERE

## TASK 16 (5 POINTS)
---
Calculate the survival function of the three first patients in the testing subset. Compare these survival functions with those obtained with the CPH models.

In [ ]:
## Plot the survival function for the three first patients in the testing subset
X_test_sel=X_test.iloc[1:4,:]

# YOUR CODE GOES HERE

What property do you observe, compared to the plot of the CPH above, in terms of the shape of the three lines?

Answer:

## TASK 17 (4 POINTS)
---
Please answer the questions below

In TASK 4, we split the dataset into two subsets for training (80%) and testing (20%). Considering the size of the dataset we use in this lab, do you think this split is a good choice？ What method or methods we could take into consideration when the dataset is too small for implementing this split?

Answer:

# TASK 18 (15 POINTS)
---
Using the decision tree code that you created in Lab 1 as a base, create your own Survival Tree algorithm, using the splitting criterion of your choice (you can use Logrank statistic or likelihood ratio statistic, as indicated in the course's slides). The algorithm should include the same hyper-parameters from Lab 1 and also min_obs_leaf, which indicates the minimum number of observations that a leaf should have. Use the training subset to train a survival tree and represent the survival functions on the three first patients from the testing subset.

In [ ]:
# YOUR CODE GOES HERE

# TASK 19 (10 POINTS) (Bonus)
---
Reimplement the survival tree algorithm from Task 18 using a different method for the splitting criteria.

In [ ]:
# YOUR CODE GOES HERE

What is a benefit of your new splitting criteria compared to the one you previously used?

Answer:


What is a drawback of your new splitting criteria compared to the one you previously used?

Answer:

## TASK 20
---
Run the following cell to finish the lab.

In [ ]:
from IPython.display import HTML

HTML('<iframe width="560" height="315" src="https://www.youtube.com/embed/WwJPoL5FLF8" frameborder="0" allow="accelerometer; autoplay; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>')

: 

Paste your colab link here:

**You are ready to submit in Gradescope!**

Please suffix your colab file with _\<JHED ID\> (It's the part before the @ symbol in your email)

e.g. Lab1_Decision_trees_myjhedID

4 easy steps to submit your lab:

1.   Click on "Share" option on top right - Click on "copy link" option. Make sure your permission is set to "Anyone on the internet with this link can view". And paste it in the cell above.
2.   Go to "File" - "Download .ipynb" and "Download .py".
3.   Export the notebook to a PDF file with all the outputs.
3.   Upload the three files (.pdf, .ipynb, .py) to Gradescope.

That's it!